# Phase 2: ML Attrition Model — Step 2.4: Model Explainability

This notebook loads the trained Logistic Regression pipeline and applies SHAP (SHapley Additive exPlanations) to explain the model's predictions globally (feature importance) and locally (individual employee explanations).

We:
1. Load the pipeline from `models/attrition_pipeline.joblib`.
2. Perform feature engineering and one-hot encoding on `employees.csv` to match the model schema.
3. Extract the scaler and classifier from the pipeline.
4. Scale features and compute SHAP values using `shap.LinearExplainer`.
5. Produce and save a global SHAP summary plot.
6. Produce and save a local SHAP force plot for a single employee (first employee in dataset).

In [1]:
import os
import joblib
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

proc_dir = os.path.join("data", "processed")
figures_dir = os.path.join("notebooks", "figures")
os.makedirs(figures_dir, exist_ok=True)

## 1. Load Data & Prepare Model Schema
We apply the exact same feature engineering, column drops, and encoding steps as in model comparison to prepare the features `X_encoded` and target `y`.

In [2]:
df_emp = pd.read_csv(os.path.join(proc_dir, "employees.csv"))

# Feature Engineering
df_emp["income_per_year_at_company"] = (df_emp["MonthlyIncome"] * 12) / (df_emp["YearsAtCompany"] + 1.0)
df_emp["years_since_last_promotion_gap"] = df_emp["YearsAtCompany"] - df_emp["YearsSinceLastPromotion"]
df_emp["overall_satisfaction_composite"] = (
    df_emp["EnvironmentSatisfaction"] 
    + df_emp["JobSatisfaction"] 
    + df_emp["RelationshipSatisfaction"] 
    + df_emp["WorkLifeBalance"]
) / 4.0
df_emp["experience_ratio"] = df_emp["YearsAtCompany"] / (df_emp["TotalWorkingYears"] + 1.0)

constant_cols = ["EmployeeCount", "Over18", "StandardHours"]
id_cols = ["EmployeeNumber"]
target_col = "Attrition"
y = df_emp[target_col].map({"Yes": 1, "No": 0})
X = df_emp.drop(columns=constant_cols + id_cols + [target_col])

# One-Hot Encoding
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
bool_cols = X_encoded.select_dtypes(include=['bool']).columns
X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)

print(f"Encoded features shape: {X_encoded.shape}")

Encoded features shape: (1470, 48)


## 2. Load Winning Pipeline & Extract Components
We load the serialized pipeline, which contains both scaling and model coefficients.

In [3]:
pipeline_path = os.path.join("models", "attrition_pipeline.joblib")
pipeline = joblib.load(pipeline_path)

scaler = pipeline.named_steps["scaler"]
model = pipeline.named_steps["classifier"]

print("Pipeline loaded successfully.")

Pipeline loaded successfully.


## 3. Scale Data & Compute SHAP Values
We scale the features using the loaded scaler, then construct a SHAP linear explainer to compute SHAP values for the entire dataset.

In [4]:
X_scaled = pd.DataFrame(scaler.transform(X_encoded), columns=X_encoded.columns)

# Compute SHAP values
explainer = shap.LinearExplainer(model, X_scaled)
shap_values = explainer(X_scaled)

print(f"SHAP Expected Value: {explainer.expected_value}")
print(f"SHAP Values Shape: {shap_values.shape}")

SHAP Expected Value: -0.9555396057480903
SHAP Values Shape: (1470, 48)


## 4. Global Feature Importance (SHAP Summary Plot)
The summary plot shows the distribution of the SHAP values for each feature, sorted by global feature importance. It visualizes how feature values impact predictions.

In [5]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_scaled, show=False)
plt.tight_layout()

summary_plot_path = os.path.join(figures_dir, "shap_summary.png")
plt.savefig(summary_plot_path, bbox_inches="tight")
print(f"Global SHAP summary plot saved to {summary_plot_path}")
plt.close()

Global SHAP summary plot saved to notebooks\figures\shap_summary.png


## 5. Local Explanation (Individual Force Plot)
We generate a force plot for the first employee in the dataset (index 0) to explain their individual probability prediction.

In [6]:
plt.figure(figsize=(12, 4))
shap.force_plot(
    explainer.expected_value,
    shap_values.values[0],
    X_scaled.iloc[0],
    matplotlib=True,
    show=False
)
plt.tight_layout()

force_plot_path = os.path.join(figures_dir, "shap_force_local.png")
plt.savefig(force_plot_path, bbox_inches="tight")
print(f"Local SHAP force plot saved to {force_plot_path}")
plt.close()

Local SHAP force plot saved to notebooks\figures\shap_force_local.png
